In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

Load Clean Dataset

In [3]:
data = pd.read_csv('../data/final_data_cleaned.csv')

In [4]:
data

,loan_amnt,funded_amnt,term,int_rate,installment,emp_length,annual_inc,zip_code,dti,delinq_2yrs,...,job_category_executive_owner,job_category_finance,job_category_mechanical_engineering,job_category_medical,job_category_nursing,job_category_other,job_category_other_engineering,job_category_sales,job_category_software_it,job_category_supervisor
0,0.050633,2500,36,0.321262,84.92,10,55000.0,109,0.01924,0.0,...,False,False,False,False,False,True,False,False,False,False
1,0.746835,30000,60,0.530763,777.23,10,90000.0,713,0.02752,0.0,...,False,False,False,False,False,True,False,False,False,False
2,0.113924,5000,36,0.492991,180.69,6,59280.0,490,0.01151,0.0,...,False,False,False,False,False,False,False,False,False,False
3,0.088608,4000,36,0.530763,146.51,10,92000.0,985,0.01774,0.0,...,False,False,False,False,False,True,False,False,False,False
4,0.746835,30000,60,0.421729,731.78,10,57250.0,212,0.02735,0.0,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260663,0.291139,12000,60,0.341511,279.72,10,58000.0,54,0.02188,0.0,...,False,False,False,False,False,True,False,False,False,False
2260664,0.291139,12000,60,0.798676,358.01,0,30000.0,971,0.02028,3.0,...,False,False,False,False,False,True,False,False,False,False
2260665,0.240506,10000,36,0.260125,332.10,10,64000.0,603,0.01396,0.0,...,False,False,False,False,False,True,False,False,False,False
2260666,0.291139,12000,60,0.628505,327.69,10,60000.0,996,0.03182,2.0,...,False,False,False,False,False,True,False,False,False,False


In [5]:
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso_forest.fit(data)

# Predict anomalies (-1 = anomaly, 1 = normal)
data["fraud_prediction"] = iso_forest.predict(data)

# Convert predictions to binary (1 = fraud, 0 = normal)
data["fraud_prediction"] = data["fraud_prediction"].apply(lambda x: 1 if x == -1 else 0)

In [6]:
loan_stacking = data.groupby(["zip_code", "loan_amnt"]).size().reset_index(name="loan_count")

# Merge back into dataset
data = data.merge(loan_stacking, on=["zip_code", "loan_amnt"], how="left")

# Flag loan stacking fraud (If more than 2 loans with the same amount in the same area)
data["loan_stacking_fraud"] = data["loan_count"].apply(lambda x: 1 if x > 2 else 0)

# Drop temporary `loan_count` column (optional)
data.drop(columns=["loan_count"], inplace=True)

In [7]:
data["zip_prefix"] = data["zip_code"].astype(str).str[:3]

# Create a list of state abbreviations (Already One-Hot Encoded)
state_columns = [col for col in data.columns if "addr_state_" in col]

# Check if the ZIP prefix belongs to a mismatched state
data["geo_fraud"] = (data[state_columns].sum(axis=1) == 0).astype(int)  # Flag if no matching state found

# Drop the temporary `zip_prefix` column
data.drop(columns=["zip_prefix"], inplace=True)

In [8]:
#80% train , 20% test
stacking_fraud = data.sample(1000).copy()
stacking_fraud["last_pymnt_d"] = np.random.randint(201901, 202412, stacking_fraud.shape[0])
stacking_fraud["loan_amnt"] *= np.random.uniform(1.2, 2.5, stacking_fraud.shape[0])
stacking_fraud["fraud_prediction"] = 1  # Mark as fraud

# Create synthetic geographical fraud data
geo_fraud = data.sample(1000).copy()

# Select random ZIP codes from actual dataset
geo_fraud["zip_code"] = np.random.choice(data["zip_code"].unique(), geo_fraud.shape[0])

# Select random states using One-Hot Encoded `addr_state`
geo_fraud_state_columns = [col for col in data.columns if "addr_state_" in col]

# Randomly assign fraudulent locations to One-Hot Encoded states
for col in geo_fraud_state_columns:
    geo_fraud[col] = np.random.choice([0, 1], geo_fraud.shape[0])  # Randomly assign states

# Mark these synthetic records as fraud
geo_fraud["fraud_prediction"] = 1  

# Combine datasets
final_dataset = pd.concat([data, stacking_fraud, geo_fraud])

# Save dataset with synthetic fraud cases
final_dataset.to_csv("../data/final_data_with_synthetic.csv", index=False)

In [9]:
data = pd.read_csv("../data/final_data_with_synthetic.csv")

# Define Features and Target
y = data["fraud_prediction"]
X = data.drop(columns=["fraud_prediction"])

# Normalize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, "../models/scaler_with_synthetic.pkl")

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Define & Train the Neural Network
class FraudDetectionModel(nn.Module):
    def __init__(self, input_size):
        super(FraudDetectionModel, self).__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# Initialize model
input_size = X_train.shape[1]
model = FraudDetectionModel(input_size)

# Train Model
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

In [12]:
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred = (y_pred > 0.5).float().numpy()  # Convert to binary predictions (1 = fraud, 0 = normal)

# Convert tensors to numpy arrays for evaluation
y_test_np = y_test_tensor.numpy()

In [13]:
print("✅ Model Evaluation Report:")
print(classification_report(y_test_np, y_pred))

# Print accuracy score
accuracy = accuracy_score(y_test_np, y_pred)
print(f"✅ Model Accuracy: {accuracy:.4f}")

# Print confusion matrix
conf_matrix = confusion_matrix(y_test_np, y_pred)
print("✅ Confusion Matrix:")
print(conf_matrix)

✅ Model Evaluation Report:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97    429527
         1.0       0.95      0.01      0.01     23007

    accuracy                           0.95    452534
   macro avg       0.95      0.50      0.49    452534
weighted avg       0.95      0.95      0.93    452534

✅ Model Accuracy: 0.9494
✅ Confusion Matrix:
[[429520      7]
 [ 22879    128]]
